In [1]:
%pip install ollama transformers sounddevice

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ollama

LLM_ID = "qwen3:1.7b"

ollama.chat(model=LLM_ID, messages=[{"role": "user", "content": "hi"}], think=False)
print(f"{LLM_ID} is ready")

qwen3:1.7b is ready


In [3]:
import phone_tools

phone_tools.DRY_RUN = False  # True = only print the adb commands (for testing without a phone)
TOOLS_BY_NAME = {tool.__name__: tool for tool in phone_tools.TOOLS}  # "open_app" -> the open_app function

In [4]:
import json

SYSTEM_PROMPT = (
    "You are a helpful voice assistant on smart glasses. "
    "The user's words come from speech recognition and may contain mistakes; guess what they meant. "
    "You can act on the user's Android phone with your tools; use a tool only when the user asks for a phone action. "
    "To call or text someone, pass their number or their name to the tool. Never make up phone numbers. "
    "Answer the user's latest message directly, in one or two short sentences."
)

history = [{"role": "system", "content": SYSTEM_PROMPT}]
MAX_TURNS = 6  # keep the last N question/answer pairs so the prompt doesn't grow forever


def tool_call_written_as_text(answer):
    """Small models sometimes write a tool call as plain text, e.g. {"name": "phone_status", "arguments": {...}},
    instead of using the tool-call format. If the answer is exactly that, turn it into a real tool call."""
    try:
        data = json.loads(answer[answer.find("{"): answer.rfind("}") + 1])
    except json.JSONDecodeError:
        return []
    if not isinstance(data, dict) or data.get("name") not in TOOLS_BY_NAME:
        return []
    function = ollama.Message.ToolCall.Function(name=data["name"], arguments=data.get("arguments") or {})
    return [ollama.Message.ToolCall(function=function)]


def run_tool(call):
    """Run the function the LLM asked for and return its result as text."""
    name, args = call.function.name, call.function.arguments
    tool = TOOLS_BY_NAME.get(name)
    try:
        result = tool(**args) if tool else f"Error: there is no tool called {name}."
    except TypeError as e:  # the LLM passed wrong or missing arguments
        result = f"Error: wrong arguments for {name}: {e}"
    print(f"   [tool] {name}({args}) -> {result}")
    return result


def ask_llm(transcript, max_new_tokens=150):
    """Send one ASR transcript to the LLM, run any tools it asks for, stream the answer and return it."""
    history.append({"role": "user", "content": transcript})
    del history[1:-(2 * MAX_TURNS + 1)]  # drop old turns, keep the system prompt

    while True:  # 1st round: the LLM answers OR asks for tools; after tools run, it answers with their results
        answer, tool_calls = "", []
        print("LLM:", end=" ", flush=True)
        for part in ollama.chat(
            model=LLM_ID,
            messages=history,
            tools=phone_tools.TOOLS,  # Ollama reads each function's name, arguments and docstring
            stream=True,
            think=False,
            options={"temperature": 0.7, "num_predict": max_new_tokens},
        ):
            print(part.message.content, end="", flush=True)
            answer += part.message.content
            tool_calls += part.message.tool_calls or []
        print()

        tool_calls = tool_calls or tool_call_written_as_text(answer)
        history.append({"role": "assistant", "content": answer.strip(), "tool_calls": tool_calls})
        if not tool_calls:
            return answer.strip()
        for call in tool_calls:
            history.append({"role": "tool", "content": run_tool(call), "tool_name": call.function.name})

In [5]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]

ask_llm("hello my name is ivan what is your name")
ask_llm("what is the capital of france")
ask_llm("and how many people live there")  # follow-up: needs the history to know "there" = Paris

LLM: My name is Assistant. How can I assist you today?
LLM: The capital of France is Paris.
LLM: The population of Paris is approximately 2.1 million.


'The population of Paris is approximately 2.1 million.'

In [6]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]

ask_llm("open youtube")
ask_llm("text my mom that I'll be home soon")  # a name, not a number: send_sms looks "mom" up in the contacts
ask_llm("how is my battery")
ask_llm("take me to Sofia airport")
ask_llm("do I have any new messages")

LLM: 
   [tool] open_app({'app_name': 'youtube'}) -> Opened youtube.
LLM: YouTube has been opened.
LLM: 
   [tool] send_sms({'contact': 'Mom', 'message': "I'll be home soon."}) -> The message to 0878151185 is ready. The user must press send.
LLM: The message to Mom has been sent. Press the send button to send it.
LLM: 
   [tool] phone_status({'item': 'battery'}) -> Battery 35%, charging.
LLM: Your battery is at 35% and it's charging.
LLM: 
   [tool] navigate_to({'destination': 'Sofia airport'}) -> Google Maps is navigating to Sofia airport.
LLM: Google Maps is navigating to Sofia airport.
LLM: 
   [tool] read_notifications({}) -> google: Google: You have a notification; 13° tomorrow · 4° cooler than today: See full forecast for Sofia (+1 more) | livesport: Bournemouth - AC Milan: ⚽ 32' Goal! [2] - 0 Tavernier M. (Penalty); Juventus - Rennes 2 - 2: Woltemade strikes again to continue Rennes-aissance (+5 more) | threads: Follow nikolay efremov (nikolay_6184): “🇧🇬”; _elena.dobreva · New: 

'You have new notifications. Here are a few highlights:  \n- Weather forecast update for tomorrow.  \n- Sports match results and recommendations.  \n- Deals and promotions.  \n- New photos and updates.  \n- Social media and app notifications.'

In [7]:
from asr_stream import LiveTranscriber

asr = LiveTranscriber(mic="Headset CMF WASAPI")

c:\Users\Lenovo\Desktop\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 651/651 [00:05<00:00, 113.88it/s]


Streaming latency: 560 ms


In [10]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]  # start a fresh conversation

try:
    for transcript in asr.utterances(pause_seconds=1.5):
        ask_llm(transcript)
        print()
except KeyboardInterrupt:
    print("\nStopped.")

Listening... speak now (press Interrupt to stop)
Hello, please open YouTube
LLM: 
   [tool] open_app({'app_name': 'youtube'}) -> Opened youtube.
LLM: YouTube has been opened.

Do I have any new messages
LLM: 
   [tool] read_notifications({'item': 'new_messages'}) -> Error: wrong arguments for read_notifications: read_notifications() got an unexpected keyword argument 'item'
LLM: The function "read_notifications" does not accept the argument "item". Please try again without specifying an item.

call Mom?
LLM: 
   [tool] find_contact({'name': 'Mom'}) -> Mom: 0878151185
LLM: Mom's phone number is 0878151185.

Brother.
LLM: 
   [tool] find_contact({'name': 'Brother'}) -> Brother Ivaylo: +359876273603; Brother Rado: +359884181781; Brother Momo: +359887618714; Brother Ivo: +359887702017; Brother Vankicha: +359890202583
LLM: Here are the phone numbers for your brothers:

- Brother Ivaylo: +359876273603  
- Brother Rado: +359884181781  
- Brother Momo: +359887618714  
- Brother Ivo: +359887702